# Решения: Типы признаков и apply на заказах

**Для преподавателя.** Секционный эталон урока и ДЗ; до сдачи ученикам не показывать.


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## Урок. 1. Три таблицы — три единицы наблюдения

In [ ]:
sizes = {"orders": len(orders), "customers": len(customers), "payments": len(payments)}
UNITS_NOTE = "Строка orders — заказ, customers — запись клиента, payments — оплата заказа. Перед join проверяем ключ и ожидаем сохранение 3500 заказов."
assert sizes == {"orders": 3500, "customers": 778, "payments": 3500}


## Урок. 2. Числовые и категориальные столбцы

In [ ]:
feature_types = {"payment_value": "numeric", "payment_type": "categorical", "customer_state": "categorical", "order_status": "categorical"}
assert feature_types["payment_value"] == "numeric"


## Урок. 3. Безопасный join заказов и оплат

In [ ]:
orders_pay = orders.merge(payments, on="order_id", how="left", validate="one_to_one")
assert len(orders_pay) == len(orders) and orders_pay["order_id"].is_unique


## Урок. 4. Функция категории оплаты

In [ ]:
def amount_band(value):
    if value <= 100: return "small"
    if value <= 300: return "mid"
    return "big"
assert amount_band(300) == "mid"


## Урок. 5. Apply к одному столбцу

In [ ]:
orders_pay["payment_band"] = orders_pay["payment_value"].apply(amount_band)
band_counts = orders_pay["payment_band"].value_counts()
assert int(band_counts.sum()) == len(orders_pay)


## Урок. 6. Apply по строке

In [ ]:
orders_pay["days_to_deliver"] = orders_pay.apply(lambda r: (r["order_delivered_customer_date"] - r["order_purchase_timestamp"]).days if pd.notna(r["order_delivered_customer_date"]) else np.nan, axis=1)
assert orders_pay["days_to_deliver"].dropna().ge(0).all()


## Урок. 7. Apply или векторизация

In [ ]:
vector_days = (orders_pay["order_delivered_customer_date"] - orders_pay["order_purchase_timestamp"]).dt.days
same_days = bool(vector_days.equals(orders_pay["days_to_deliver"]))
VECTOR_NOTE = "Apply удобен для сложного правила из нескольких полей строки, но разность datetime уже векторизована. Векторная запись короче, обычно быстрее и лучше показывает смысл операции."
assert same_days and len(VECTOR_NOTE) >= 160


## Урок. 8. Карточка признака

In [ ]:
FEATURE_CARD = "Источник: две даты orders. Тип: числовой, дни. Правило: delivery minus purchase; для недоставленного заказа пропуск сохраняется. Риск: признак появляется после покупки и может быть недоступен в момент раннего решения; отрицательные значения означают ошибку данных. Перед моделью проверяем момент доступности."
assert len(FEATURE_CARD) >= 240


## ДЗ. 1. Диапазон дат

In [ ]:
min_date=orders["order_purchase_timestamp"].min(); max_date=orders["order_purchase_timestamp"].max()
span_days=(max_date-min_date).days
assert span_days > 500


## ДЗ. 2. Доли типов оплаты

In [ ]:
pay_share=payments["payment_type"].value_counts(normalize=True); top_payment=pay_share.idxmax()
assert top_payment=="credit_card"


## ДЗ. 3. Признак is_card

In [ ]:
payments["is_card"]=payments["payment_type"].apply(lambda x: int(x=="credit_card"))
assert set(payments["is_card"])=={0,1}


## ДЗ. 4. Challenge: универсальный биннер

In [ ]:
def make_binner(low, high):
    return lambda x: "small" if x<=low else ("mid" if x<=high else "big")
binner=make_binner(100,300)
assert binner(301)=="big"


## ДЗ. 5. Challenge: инженерная записка

In [ ]:
APPLY_NOTE="Apply оставляем для составного построчного правила, когда ветвление использует несколько полей. Векторную операцию выбираем для арифметики столбцов: она проще и быстрее. Для биннинга отдельно тестируем значения на границах 100 и 300, а также пропуски, чтобы контракт категорий не менялся незаметно."
assert len(APPLY_NOTE)>=220
